# Vectors and Embeddings: How Words Become Geometry

An introduction to the idea at the heart of large language models: **a word can be turned into a list of numbers (a vector), and the *geometry* of those vectors can carry meaning.**

This notebook is inspired by the 3Blue1Brown animations on word vectors. We build everything from scratch with plain NumPy, using a tiny hand-made vocabulary so that every number can be checked by eye.

**How to use it.** Run the cells top to bottom. Cells with sliders and dropdowns are meant to be played with, both in the lecture and by students. Each section ends with a short "what to notice" prompt.

**Plan**

1. Vectors as arrows (adding, scaling)
2. Measuring similarity: dot product and cosine similarity
3. A toy embedding where we know what every axis means
4. Nearest neighbours: "which words are close to this word?"
5. The famous `king - man + woman ≈ queen`
6. Meaning as a *direction* in space
7. Why real embeddings do not have readable axes (rotation experiment)
8. *(Optional)* The same ideas on real GloVe vectors
9. Exercises and discussion questions

In [ ]:
# ---- Setup: run this cell first ----
import numpy as np                 # arrays and basic vector maths
import matplotlib.pyplot as plt    # plotting
import ipywidgets as widgets       # sliders and dropdowns
from ipywidgets import interact

%matplotlib inline
plt.rcParams["figure.figsize"] = (6, 6)   # default plot size

## 1. Vectors as arrows

A **vector** is just a list of numbers. In 2D, `[3, 1]` means "go 3 steps right and 1 step up". We can draw it as an arrow from the origin.

Two operations matter for everything that follows:

* **Adding** vectors: add them component by component (put the arrows tip to tail).
* **Scaling** a vector: multiply every component by a number (stretch or flip the arrow).

In [ ]:
# Two vectors in 2D, stored as NumPy arrays so that + and * work component by component.
a = np.array([3.0, 1.0])
b = np.array([1.0, 2.0])

print("a       =", a, "  shape:", a.shape)   # shape (2,) means: a list of 2 numbers
print("b       =", b)
print("a + b   =", a + b)       # [3+1, 1+2] = [4, 3]
print("2 * a   =", 2 * a)       # [6, 2]: same direction, twice as long
print("-1 * a  =", -1 * a)      # [-3, -1]: flipped to point the opposite way

In [ ]:
# A small helper to draw an arrow from `start` to `start + vector`.
# We keep it in its own cell so the later cells can stay short.
def draw_arrow(ax, vector, color, label, start=(0, 0), style="-"):
    tip_x = start[0] + vector[0]
    tip_y = start[1] + vector[1]
    # annotate with empty text is the simplest way to get an arrow in matplotlib
    ax.annotate("", xy=(tip_x, tip_y), xytext=start,
                arrowprops=dict(arrowstyle="->", color=color, lw=2.5, linestyle=style))
    ax.text(tip_x * 1.05, tip_y * 1.05, label, color=color, fontsize=13)

# A helper to make an empty square plot with axes through the origin.
def make_axes(limit=6):
    fig, ax = plt.subplots()
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    ax.set_aspect("equal")
    ax.axhline(0, color="gray", lw=0.8)
    ax.axvline(0, color="gray", lw=0.8)
    ax.grid(alpha=0.3)
    return fig, ax

In [ ]:
# INTERACTIVE: drag the sliders to change vectors a (blue) and b (orange).
# The green arrow is a + b: we draw b starting at the tip of a (tip to tail).
def show_vector_addition(a_x=3.0, a_y=1.0, b_x=1.0, b_y=2.0):
    a = np.array([a_x, a_y])
    b = np.array([b_x, b_y])
    fig, ax = make_axes(8)
    draw_arrow(ax, a, "tab:blue", "a")
    draw_arrow(ax, b, "tab:orange", "b")
    draw_arrow(ax, b, "tab:orange", "", start=a, style="--")   # b moved to the tip of a
    draw_arrow(ax, a + b, "tab:green", "a + b")
    ax.set_title("Vector addition: a + b = " + str(a + b))
    plt.show()

interact(show_vector_addition,
         a_x=(-5.0, 5.0, 0.5), a_y=(-5.0, 5.0, 0.5),
         b_x=(-5.0, 5.0, 0.5), b_y=(-5.0, 5.0, 0.5));

**What to notice.** The green arrow is the diagonal of the parallelogram made by `a` and `b`. Try making `b` point opposite to `a`: the sum shrinks. Remember this picture: in section 5, "adding word meanings" is exactly this operation, just in more than 2 dimensions.

## 2. Measuring similarity: dot product and cosine similarity

If words are vectors, "similar meaning" should become "similar direction". We need a number that says how well two arrows point the same way.

The **dot product** multiplies matching components and adds them up:

`u · v = u[0]*v[0] + u[1]*v[1] + ...`

The **cosine similarity** divides by the lengths, so only the *direction* matters:

* `+1` : pointing the same way
* `0` : at right angles (unrelated)
* `-1` : pointing opposite ways

In [ ]:
# We write these with explicit loops first, so you can trace every step by hand.
# (NOTE: np.dot does the same thing faster; we use a loop for clarity.)

def dot_product(u, v):
    total = 0.0
    for i in range(len(u)):          # go through each component
        total = total + u[i] * v[i]  # multiply matching components, add to running total
    return total

def length(u):
    # The length of an arrow is the square root of (u dot u): Pythagoras in any dimension.
    return dot_product(u, u) ** 0.5

def cosine_similarity(u, v):
    denominator = length(u) * length(v)
    if denominator == 0:             # a zero vector has no direction, so call the similarity 0
        return 0.0
    return dot_product(u, v) / denominator

# Hand-checkable examples:
print("same direction  :", cosine_similarity([1, 0], [5, 0]))    # 1.0  (length does not matter)
print("right angle     :", cosine_similarity([1, 0], [0, 3]))    # 0.0
print("opposite        :", cosine_similarity([1, 0], [-2, 0]))   # -1.0
print("in between      :", cosine_similarity([1, 0], [1, 1]))    # 0.707 = cos(45 degrees)

In [ ]:
# INTERACTIVE: change the two vectors and watch the numbers.
def show_similarity(a_x=3.0, a_y=1.0, b_x=1.0, b_y=2.0):
    a = np.array([a_x, a_y])
    b = np.array([b_x, b_y])
    fig, ax = make_axes(6)
    draw_arrow(ax, a, "tab:blue", "a")
    draw_arrow(ax, b, "tab:orange", "b")
    cos_sim = cosine_similarity(a, b)
    angle = np.degrees(np.arccos(np.clip(cos_sim, -1, 1)))  # clip guards against rounding error
    ax.set_title("dot = %.2f    cosine = %.2f    angle = %.0f degrees"
                 % (dot_product(a, b), cos_sim, angle))
    plt.show()

interact(show_similarity,
         a_x=(-5.0, 5.0, 0.5), a_y=(-5.0, 5.0, 0.5),
         b_x=(-5.0, 5.0, 0.5), b_y=(-5.0, 5.0, 0.5));

**What to notice.** Stretch one arrow without turning it: the *dot product* changes but the *cosine* does not. Cosine similarity is the measure we will use for "how similar are two words?".

## 3. A toy embedding where we know what every axis means

Real models *learn* their vectors from huge amounts of text. To see the idea clearly, we first **design** a small embedding by hand. Each word gets 5 numbers, and we decide what each position means:

| position | meaning | values |
|---|---|---|
| 0 | royalty | 0 = ordinary, 1 = royal |
| 1 | gender | +1 = male, -1 = female, 0 = neutral |
| 2 | age | 0 = young, 1 = adult |
| 3 | animal | 0 = person, 1 = animal |
| 4 | food | 0 = not food, 1 = food |

In [ ]:
# The embedding table: a dictionary mapping each word to its vector.
# Order of numbers: [royalty, gender, age, animal, food]
dimension_names = ["royalty", "gender", "age", "animal", "food"]

embeddings = {
    # people
    "king":     np.array([1.0,  1.0, 0.8, 0.0, 0.0]),
    "queen":    np.array([1.0, -1.0, 0.8, 0.0, 0.0]),
    "prince":   np.array([0.8,  1.0, 0.2, 0.0, 0.0]),
    "princess": np.array([0.8, -1.0, 0.2, 0.0, 0.0]),
    "man":      np.array([0.0,  1.0, 0.8, 0.0, 0.0]),
    "woman":    np.array([0.0, -1.0, 0.8, 0.0, 0.0]),
    "boy":      np.array([0.0,  1.0, 0.2, 0.0, 0.0]),
    "girl":     np.array([0.0, -1.0, 0.2, 0.0, 0.0]),
    # animals (nearly gender neutral, so gender is close to 0)
    "dog":      np.array([0.0,  0.1, 0.8, 1.0, 0.0]),
    "cat":      np.array([0.0, -0.1, 0.8, 1.0, 0.0]),
    "puppy":    np.array([0.0,  0.1, 0.2, 1.0, 0.0]),
    "kitten":   np.array([0.0, -0.1, 0.2, 1.0, 0.0]),
    # food
    "apple":    np.array([0.0,  0.0, 0.1, 0.0, 1.0]),
    "banana":   np.array([0.0,  0.0, 0.1, 0.0, 0.9]),
    "bread":    np.array([0.0,  0.0, 0.5, 0.0, 0.8]),
}

# Print the whole table so students can read every number.
print("%-10s" % "" + "".join("%9s" % name for name in dimension_names))
for word in embeddings:
    print("%-10s" % word + "".join("%9.1f" % x for x in embeddings[word]))

In [ ]:
# INTERACTIVE: we cannot draw 5 dimensions, so choose which 2 axes to look at.
# Each choice is a "shadow" (projection) of the 5D cloud onto a flat plane.
def plot_two_axes(x_axis="gender", y_axis="royalty"):
    x_index = dimension_names.index(x_axis)
    y_index = dimension_names.index(y_axis)
    fig, ax = plt.subplots(figsize=(7, 6))
    for word in embeddings:
        x = embeddings[word][x_index]
        y = embeddings[word][y_index]
        ax.scatter(x, y, color="tab:blue")
        ax.text(x + 0.03, y + 0.03, word, fontsize=11)
    ax.set_xlabel(x_axis)
    ax.set_ylabel(y_axis)
    ax.grid(alpha=0.3)
    ax.set_title("Toy embedding seen through %s vs %s" % (x_axis, y_axis))
    plt.show()

interact(plot_two_axes, x_axis=dimension_names, y_axis=dimension_names);

**What to notice.** With `gender` vs `royalty`, king and queen sit at the same height but mirror each other left/right. With `animal` vs `age`, puppies and kittens cluster away from dogs and cats. Some words land on top of each other in certain views: that is the "shadow" effect of dropping dimensions, and one reason real models use hundreds or thousands of dimensions.

## 4. Nearest neighbours: which words are close to this word?

Once words are vectors, "find similar words" becomes "find the vectors with the highest cosine similarity". This is the simplest possible use of an embedding.

In [ ]:
# Score every word in the vocabulary against a query vector and keep the best few.
def nearest_words(query_vector, how_many=3, exclude=()):
    scored_words = []
    for word in embeddings:
        if word in exclude:                      # sometimes we want to skip the query words themselves
            continue
        score = cosine_similarity(query_vector, embeddings[word])
        scored_words.append((score, word))       # store as (score, word) so sorting uses the score
    scored_words.sort(reverse=True)              # highest similarity first
    return scored_words[:how_many]

# Try it: what is closest to "king"? (Expect king itself first with score 1.0.)
for score, word in nearest_words(embeddings["king"], how_many=4):
    print("%-10s %.3f" % (word, score))

In [ ]:
# INTERACTIVE: pick a word and see its neighbours.
def show_neighbours(word="king", how_many=5):
    print("Closest words to '%s':" % word)
    for score, neighbour in nearest_words(embeddings[word], how_many=how_many + 1, exclude=(word,)):
        print("  %-10s %6.3f  %s" % (neighbour, score, "#" * int(max(score, 0) * 30)))

interact(show_neighbours, word=list(embeddings.keys()), how_many=(1, 10));

**What to notice.** Neighbours of `king` are other royal, adult, male-ish words. Neighbours of `puppy` are other young animals. Nobody told the computer these categories; they fall out of the geometry.

## 5. The famous example: `king - man + woman ≈ queen`

Adding and subtracting vectors lets us do **arithmetic on meaning**:

* `king - man` removes "male adult person", leaving roughly "royalty".
* `+ woman` adds "female adult person" back.

The result should land near `queen`. Let us do it step by step.

In [ ]:
# Step by step, printing every intermediate vector.
print("dimensions:", dimension_names)
print("king            =", embeddings["king"])
print("man             =", embeddings["man"])
print("woman           =", embeddings["woman"])

king_minus_man = embeddings["king"] - embeddings["man"]
print("king - man      =", king_minus_man, " <- only royalty is left")

result = king_minus_man + embeddings["woman"]
print("king - man + woman =", result)

# Which word is closest to this new vector? We exclude the three input words,
# otherwise "king" or "woman" might win trivially.
print()
for score, word in nearest_words(result, how_many=3, exclude=("king", "man", "woman")):
    print("%-10s %.3f" % (word, score))

In [ ]:
# INTERACTIVE: try any analogy "A - B + C".
# The "messiness" slider adds random noise to every vector, to mimic the fact that
# real embeddings are learned and imperfect. Watch how long the analogy survives.

# One fixed random wiggle per word (seeded so the picture is repeatable).
random_generator = np.random.RandomState(0)
noise_vectors = {word: random_generator.normal(size=5) for word in embeddings}

def show_analogy(A="king", B="man", C="woman", messiness=0.0):
    # Build a noisy copy of the embedding for this slider setting.
    noisy = {word: embeddings[word] + messiness * noise_vectors[word] for word in embeddings}

    result = noisy[A] - noisy[B] + noisy[C]

    # Rank all words (except the inputs) against the result vector.
    ranked = []
    for word in noisy:
        if word in (A, B, C):
            continue
        ranked.append((cosine_similarity(result, noisy[word]), word))
    ranked.sort(reverse=True)

    print("%s - %s + %s  ~  %s" % (A, B, C, ranked[0][1].upper()))
    for score, word in ranked[:3]:
        print("   %-10s %.3f" % (word, score))

    # Plot in the (gender, royalty) plane: it shows the arithmetic as arrows.
    gender_i, royalty_i = 1, 0
    fig, ax = plt.subplots(figsize=(7, 6))
    for word in noisy:
        x, y = noisy[word][gender_i], noisy[word][royalty_i]
        ax.scatter(x, y, color="lightgray")
        ax.text(x + 0.03, y + 0.03, word, fontsize=9, color="gray")
    for word, colour in ((A, "tab:blue"), (B, "tab:red"), (C, "tab:green")):
        ax.scatter(noisy[word][gender_i], noisy[word][royalty_i], color=colour, s=80)
    ax.scatter(result[gender_i], result[royalty_i], marker="*", s=300, color="gold",
               edgecolor="black", zorder=5, label="result")
    ax.legend()
    ax.set_xlabel("gender"); ax.set_ylabel("royalty"); ax.grid(alpha=0.3)
    ax.set_title("Blue=A, red=B, green=C, star=result")
    plt.show()

words = list(embeddings.keys())
interact(show_analogy, A=words, B=words, C=words,
         messiness=widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05));

**What to notice.**

* With `messiness = 0` the answer is exact, because we built the table so that it would be.
* Raise `messiness` and see when the answer starts to fail. Real embeddings are like the noisy version: analogies work often, not always.
* Try other analogies: `puppy - dog + cat`, `princess - girl + boy`, `apple - ... `. Which ones make sense, and which give nonsense?

## 6. Meaning as a direction in space

The analogy works because **"male to female" is a direction** in the space. Moving from `man` to `woman` is the same step as moving from `king` to `queen` or `boy` to `girl`.

We can estimate that direction by averaging several (male - female) differences.

In [ ]:
# Average the difference vectors of several male/female pairs to get one "gender direction".
male_female_pairs = [("king", "queen"), ("man", "woman"), ("prince", "princess"), ("boy", "girl")]

sum_of_differences = np.zeros(5)                  # start with a zero vector
for male_word, female_word in male_female_pairs:
    difference = embeddings[male_word] - embeddings[female_word]
    print("%-8s - %-9s = %s" % (male_word, female_word, difference))
    sum_of_differences = sum_of_differences + difference

gender_direction = sum_of_differences / len(male_female_pairs)   # the average difference
gender_direction = gender_direction / length(gender_direction)   # rescale to length 1 (a unit vector)
print()
print("gender direction (unit length) =", gender_direction)

In [ ]:
# Project every word onto the gender direction: the dot product with a unit vector
# tells us "how far along that direction is this word?".
scores = []
for word in embeddings:
    scores.append((dot_product(embeddings[word], gender_direction), word))
scores.sort()

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh([w for s, w in scores], [s for s, w in scores],
        color=["tab:red" if s < 0 else "tab:blue" for s, w in scores])
ax.set_xlabel("position along the gender direction (negative = female, positive = male)")
ax.axvline(0, color="black", lw=0.8)
plt.show()

In [ ]:
# INTERACTIVE: take a word and push it along the gender direction.
# Negative push = more female, positive push = more male.
def push_along_gender(word="king", push=0.0):
    moved_vector = embeddings[word] + push * gender_direction
    print("Starting word:", word, "   push:", push)
    print("Closest word to the moved vector (excluding the start word):")
    for score, neighbour in nearest_words(moved_vector, how_many=3, exclude=(word,)):
        print("   %-10s %.3f" % (neighbour, score))

interact(push_along_gender, word=list(embeddings.keys()),
         push=widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.25));

**What to notice.** Start at `king` and push by about `-2`: you arrive near `queen`. Start at `boy` and push the same amount: `girl`. One direction, many words. This is the geometric idea behind "the model has a concept of gender": it is a direction, not a single neuron.

## 7. Real embeddings do not have readable axes

In our toy table, position 1 literally means "gender". In a **learned** embedding, nobody assigns meanings to positions. The model could equally well store the same information in any rotated coordinate system, and it would work just as well.

Let us prove that by spinning our whole embedding with a random rotation. If meaning really lives in the *geometry* (angles and distances), nothing should break.

In [ ]:
# Make a random rotation matrix in 5D.
# QR decomposition of a random matrix gives a matrix Q whose columns are perpendicular
# unit vectors: multiplying by Q rotates (and possibly mirrors) space without stretching it.
rng = np.random.RandomState(42)
random_matrix = rng.normal(size=(5, 5))
rotation, _ = np.linalg.qr(random_matrix)
print("rotation matrix shape:", rotation.shape)

# Apply it to every word vector.
rotated_embeddings = {}
for word in embeddings:
    rotated_embeddings[word] = rotation @ embeddings[word]   # matrix times vector -> new vector, same length 5

print()
print("king, original :", np.round(embeddings["king"], 2))
print("king, rotated  :", np.round(rotated_embeddings["king"], 2), " <- the axes no longer mean anything")

In [ ]:
# Compare cosine similarities before and after rotation for several pairs.
pairs_to_check = [("king", "queen"), ("king", "man"), ("dog", "puppy"), ("apple", "banana"), ("king", "apple")]
print("%-18s %10s %10s" % ("pair", "original", "rotated"))
for first, second in pairs_to_check:
    before = cosine_similarity(embeddings[first], embeddings[second])
    after = cosine_similarity(rotated_embeddings[first], rotated_embeddings[second])
    print("%-18s %10.3f %10.3f" % (first + "/" + second, before, after))

# The analogy still works in the rotated space, too.
result = rotated_embeddings["king"] - rotated_embeddings["man"] + rotated_embeddings["woman"]
best_score, best_word = -2, None
for word in rotated_embeddings:
    if word in ("king", "man", "woman"):
        continue
    score = cosine_similarity(result, rotated_embeddings[word])
    if score > best_score:
        best_score, best_word = score, word
print()
print("rotated space: king - man + woman ~", best_word)

**What to notice.** Every similarity is identical, and the analogy still works. Rotations preserve lengths and angles, so they preserve everything that matters. This is why we say embeddings capture meaning through **relationships between vectors**, not through individual coordinates, and why interpreting a single dimension of a real model is usually meaningless.

## 8. (Optional) The same ideas on real word vectors

Everything above used 5 hand-made dimensions. Real embeddings such as **GloVe** are learned from billions of words and have 50 to 300 dimensions. This section downloads a small GloVe model (about 66 MB), so it needs an internet connection. It works well in Google Colab.

If the download fails, skip the section: nothing later depends on it.

In [ ]:
# Install and load the real vectors (this may take a minute the first time).
# In Colab/Jupyter you may need to run:  !pip install gensim
try:
    import gensim.downloader as api
    glove = api.load("glove-wiki-gigaword-50")   # 400,000 words, 50 numbers each
    print("Loaded. Vocabulary size:", len(glove.index_to_key))
    print("Vector for 'king' has", len(glove["king"]), "numbers. First five:", glove["king"][:5])
except Exception as error:
    glove = None
    print("Could not load GloVe (" + str(error) + "). Skip this section.")

In [ ]:
# The same analogy on real vectors. gensim does: positive words added, negative subtracted,
# then finds the nearest words by cosine similarity (and skips the input words).
if glove is not None:
    print("king - man + woman:")
    for word, score in glove.most_similar(positive=["king", "woman"], negative=["man"], topn=5):
        print("   %-12s %.3f" % (word, score))

    print("\nparis - france + italy:")
    for word, score in glove.most_similar(positive=["paris", "italy"], negative=["france"], topn=5):
        print("   %-12s %.3f" % (word, score))

In [ ]:
# INTERACTIVE: type any word and see its neighbours in real GloVe space.
if glove is not None:
    def real_neighbours(word="university"):
        if word not in glove:
            print("'%s' is not in the vocabulary (try lowercase)." % word)
            return
        for neighbour, score in glove.most_similar(word, topn=8):
            print("   %-14s %.3f" % (neighbour, score))
    interact(real_neighbours, word=widgets.Text(value="university"));

In [ ]:
# Squash 50 dimensions down to 2 so we can draw them (PCA, written out in a few lines).
# PCA finds the 2 directions along which the words spread out the most.
if glove is not None:
    plot_words = ["king", "queen", "man", "woman", "prince", "princess",
                  "paris", "france", "london", "england", "rome", "italy",
                  "dog", "cat", "horse", "apple", "banana", "orange"]
    vectors = np.array([glove[w] for w in plot_words])        # shape (18, 50)
    centred = vectors - vectors.mean(axis=0)                  # move the cloud's centre to the origin
    _, _, directions = np.linalg.svd(centred, full_matrices=False)
    flat = centred @ directions[:2].T                          # shape (18, 2): coordinates along top 2 directions

    fig, ax = plt.subplots(figsize=(9, 7))
    for i, word in enumerate(plot_words):
        ax.scatter(flat[i, 0], flat[i, 1], color="tab:blue")
        ax.text(flat[i, 0] + 0.05, flat[i, 1] + 0.05, word)
    ax.set_title("Real GloVe vectors, flattened from 50D to 2D")
    ax.grid(alpha=0.3)
    plt.show()

**What to notice.** Countries, capitals, animals, fruit and royalty form groups, and relationships such as `france -> paris` and `italy -> rome` appear as roughly parallel arrows. Compare with our toy version: real analogies are less exact, and nobody designed the axes.

## 9. Exercises

Solutions are in the last cell of the notebook: try first!

In [ ]:
# EXERCISE 1: add a new word to the embedding.
# Order: [royalty, gender, age, animal, food].
# Add "duchess": a royal, adult, female person. Then print its nearest neighbours.
# Hint: a duchess is slightly less royal than a queen. Use 0.6.
embeddings["duchess"] = ...      # YOUR CODE HERE (a np.array of 5 numbers)

# Uncomment after filling in the vector:
# for score, word in nearest_words(embeddings["duchess"], how_many=4, exclude=("duchess",)):
#     print(word, round(score, 3))

In [ ]:
# EXERCISE 2: predict, then check. What should  puppy - dog + cat  be?
# Write your guess in a comment BEFORE you run the code.
my_guess = ...                   # YOUR GUESS HERE (a word as a string)

result = ...                     # YOUR CODE HERE: compute the vector puppy - dog + cat
# Then find the nearest word excluding "puppy", "dog", "cat" using nearest_words(...)

In [ ]:
# EXERCISE 3: build an "age direction" (adult minus young) from these pairs:
# (king, prince), (queen, princess), (man, boy), (woman, girl), (dog, puppy), (cat, kitten).
# Then start from "puppy" and push along the direction. Which word do you reach?
# Hint: copy the structure of the gender direction code in section 6.
age_direction = ...              # YOUR CODE HERE (unit vector)

In [ ]:
# EXERCISE 4: Euclidean distance vs cosine similarity.
# Imagine a very frequent word whose vector is the same direction as "king" but 3 times longer.
# Compute BOTH the Euclidean distance (length of the difference) and the cosine similarity
# between embeddings["king"] and 3 * embeddings["king"]. Which measure says "same meaning"?
euclidean_distance = ...         # YOUR CODE HERE
cosine_value = ...               # YOUR CODE HERE

## Discussion questions

1. Why do we use cosine similarity rather than plain distance? What would go wrong for a very frequent word whose vector is long?
2. In the toy table we chose 5 dimensions by hand. What important aspects of word meaning are missing (verbs, tense, sentiment, ...)? How many dimensions might you need?
3. The rotation experiment shows individual coordinates are meaningless. How, then, could researchers find out what a direction in a real model represents?
4. In these embeddings each word has **one** vector. What happens with "bank" (river bank or money bank)? This limitation is exactly what **attention** and contextual embeddings in transformers solve, and it is the topic of the next session.

---
## Solutions (instructors: hide this cell when distributing)

In [ ]:
# ---- SOLUTIONS ----

# Exercise 1
embeddings["duchess"] = np.array([0.6, -1.0, 0.8, 0.0, 0.0])
print("Exercise 1: neighbours of duchess")
for score, word in nearest_words(embeddings["duchess"], how_many=4, exclude=("duchess",)):
    print("   %-10s %.3f" % (word, score))

# Exercise 2: puppy - dog removes "adult" and keeps "young"; + cat then adds the cat part.
# Expected: kitten.
result = embeddings["puppy"] - embeddings["dog"] + embeddings["cat"]
print("\nExercise 2: puppy - dog + cat ~", nearest_words(result, how_many=1, exclude=("puppy", "dog", "cat"))[0][1])

# Exercise 3: adult-minus-young direction
age_pairs = [("king", "prince"), ("queen", "princess"), ("man", "boy"),
             ("woman", "girl"), ("dog", "puppy"), ("cat", "kitten")]
total = np.zeros(5)
for adult_word, young_word in age_pairs:
    total = total + (embeddings[adult_word] - embeddings[young_word])
age_direction = total / len(age_pairs)
age_direction = age_direction / length(age_direction)

# Pushing "puppy" along the age direction. The size of the push needed is about the
# adult-minus-young gap (~0.6 in the age axis), so we try pushes and look at the neighbours.
print("\nExercise 3: puppy pushed along the age direction")
for push in (0.0, 0.3, 0.6, 0.9):
    moved = embeddings["puppy"] + push * age_direction
    best = nearest_words(moved, how_many=1, exclude=("puppy",))[0][1]
    print("   push %.1f -> %s" % (push, best))

# Exercise 4: same direction, different length.
longer_king = 3 * embeddings["king"]
print("\nExercise 4: king vs 3*king")
print("   euclidean distance = %.3f  (large: says 'very different')" % length(embeddings["king"] - longer_king))
print("   cosine similarity  = %.3f  (exactly 1: says 'same direction')" % cosine_similarity(embeddings["king"], longer_king))